In [2]:
import pandas as pd
df = pd.read_csv('data/train/train.csv')
# df = pd.read_csv('qwen2.5_sft_result.csv')

In [3]:
from vllm import LLM, SamplingParams
# import torch._dynamo
# torch._dynamo.config.suppress_errors = True

llm = LLM(model="Qwen2.5-1.5B-merged")

# llm = LLM(model="Qwen/Qwen2.5-14B-Instruct",gpu_memory_utilization=0.7)


sampling_params = SamplingParams(temperature=0.1,
    max_tokens=2048,
    repetition_penalty=1.05)

tokenizer = llm.get_tokenizer()

INFO 03-02 07:39:02 llm_engine.py:213] Initializing an LLM engine (v0.6.0) with config: model='Qwen2.5-1.5B-merged', speculative_config=None, tokenizer='Qwen2.5-1.5B-merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=Qwen2.5-1.5B-merged, use_v2_block_manager=False, num_scheduler_steps=1, enable_prefix_caching=False, use_async_output_proc=True)
INFO 

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-02 07:39:05 model_runner.py:926] Loading model weights took 2.8875 GB
INFO 03-02 07:39:07 gpu_executor.py:122] # GPU blocks: 34035, # CPU blocks: 9362
INFO 03-02 07:39:12 model_runner.py:1217] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 03-02 07:39:12 model_runner.py:1221] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 03-02 07:39:29 model_runner.py:1335] Graph capturing finished in 17 secs.


In [4]:
df['text'] = df['title'] + '\n' + df['paragraph']

In [5]:
from langchain import PromptTemplate



system_prompt = """Please summarize the documentation provided in 3 lines.
Also, please extract the top five key phrases. See template for the answer format.
The summary must be written in the korean
<template>
summary
- summarize 1
- summarize 2
- summarize 3

key phrases
[key phrase1, key phrase2, key phrase3, key phrase4, key phrase5]
</template>

docs:
{docs}"""

texts = []

for doc in list(df['text']):

    prompt = PromptTemplate(
    input_variables=["docs"],  # 동적 변수 목록
    template=system_prompt  # 템플릿 텍스트
    )

    # 프롬프트 생성
    final_prompt = prompt.format(docs=doc)
    messages = [{'role':'system','content':'you are a helpful assistant'}, {'role':'user','content':final_prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True)
    texts.append(text)


In [6]:
outputs = llm.generate(texts, sampling_params)

Processed prompts: 100%|██████████| 10000/10000 [35:05<00:00,  4.75it/s, est. speed input: 2858.25 toks/s, output: 1956.74 toks/s] 


In [7]:
result = [output.outputs[0].text for output in outputs]

In [8]:
df['qwen2.5_result'] = result

In [9]:
df.to_csv('data/train/train_dpo.csv',index=False)

In [11]:
df.tail(3)

,Unnamed: 0,id,title,field,paragraph,terminology,source_file,token_count,text,results,qwen2.5_result
9997,9997,1037_2842,자원순환기본법 전부개정법률안(송옥주의원등10인),NaN,안 제13조는 순환경제로의 전환 추세를 파악하기 위해 환경부장관이 국내총생산액 통계...,"환경부장관, 국내총생산액, 통계조사, 통계청장, 승인",17253_환경노동위원회_21_2111207_001.json,642,자원순환기본법 전부개정법률안(송옥주의원등10인)\n안 제13조는 순환경제로의 전환 ...,```template\nsummary\n- 자원순환기본법 개정안은 순환경제 전환 추...,### 요약\n- **순환경제로의 전환 추세를 파악하기 위해 환경부장관이 국내총생산...
9998,9998,1037_4326,도시재생 활성화 및 지원에 관한 특별법 일부개정법률안(권칠승의원 등 11인),"도시재생 활성화 및 지원에 관한 특별법,도시 및 주거환경정비법,신행정수도 후속대책을...",동 개정안은 현행법의 어려운 한자식 용어인 ‘구거’를 어문 규정에 맞는 표현인 ‘도...,"한자식 용어, 구거, 도랑, 국토교통부, 공간정보관리법",05365_국토교통위원회_20_2013929_001.json,512,도시재생 활성화 및 지원에 관한 특별법 일부개정법률안(권칠승의원 등 11인)\n동 ...,```template\nsummary\n- 도시재생 활성화 및 지원에 관한 특별법 ...,summary\n- 도시재생 활성화 및 지원에 관한 특별법 일부개정법률안은 현행법의...
9999,9999,1037_3575,형법 일부개정법률안(강창일의원 등 20인),형법,개정안은 강간죄 등을 사람의 의사에 반하여 간음하는 경우 등으로 구성요건을 변경하면...,"가중처벌, 행위수단, 행위수단, 위계, 위력, 폭행, 협박",32325_법제사법위원회_20_2012564_003.json,719,형법 일부개정법률안(강창일의원 등 20인)\n개정안은 강간죄 등을 사람의 의사에 반...,"```template\nsummary\n- 강간죄 등의 구성요건을 변경하고, 사람의...",summary\n- 개정안은 사람의 저항을 현저히 곤란하게 하는 폭행 또는 협박으로...
